# FIRST ATTEMPT

## Imports

In [ ]:
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import sklearn
import imblearn
import keras
import random
import tensorflow as tf

seed = 7
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

# Libraries for splitting, scaling, encoding and feature selection
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier


# Libraries for models
from sklearn.svm import SVC
from sklearn.naive_bayes import BernoulliNB
from sklearn import tree
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from keras.models import Sequential
from keras.layers import Dense
from keras.utils import to_categorical


from sklearn import metrics
from sklearn.model_selection import cross_val_score

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Settings
# pd.set_option('display.max_columns', None)
# import sys
# #np.set_printoptions(threshold=np.nan)
# np.set_printoptions(threshold=sys.maxsize)
# np.set_printoptions(precision=3)
# sns.set(style="darkgrid")
# plt.rcParams['axes.labelsize'] = 14
# plt.rcParams['xtick.labelsize'] = 12
# plt.rcParams['ytick.labelsize'] = 12

In [ ]:
#command used to mount drive
from google.colab import drive
drive.mount('/content/drive')



## Load and Preview The Data
#### I will be starting with the IDS 2025 data which is not partitioned yet
#### I will then preview the dataset, show its shape and summary statistics

In [ ]:
IDS_dataset = pd.read_excel('/content/drive/MyDrive/solutions/IDS2025.xlsx')
print(IDS_dataset.head())
print("The dataset has {} rows and {} columns".format(IDS_dataset.shape[0],IDS_dataset.shape[1]))
IDS_dataset.describe()

## EDA
#### 1. Check for class imbalance

In [ ]:
# Show number of rows for each class
IDS_dataset["newLabel"].value_counts()

# # From the above results, some attack classes make up a very small proportion of the dataset
# # For reporting purposes, I would like to get the percentages of each type of attack

# IDS_dataset["newLabel"].value_counts(normalize=True) * 100


#### 2. Data Cleaning - Missing Values

In [ ]:

missing = IDS_dataset.isnull().sum()
print(missing[missing > 0])

In [ ]:
# There are 77 missinng values which I have to fix so I can scale the numeric values without errors

# # These values are in Flow Bytess so let me get a preview of the column
# print(IDS_dataset["Flow Bytess"].head(20))

# To know for sure if this is random, I will have to check if this is particular to a certain type of attack
IDS_dataset[IDS_dataset['Flow Bytess'].isnull()]['newLabel'].value_counts()
# From the result, the null values are mostly concentrated on the DoS/DDoS Attack type

# # Because it is DDoS, check the attack duration. DDoS usually causes really short flow duration
# IDS_dataset[IDS_dataset['Flow Bytess'].isnull()][['Flow Duration', 'newLabel']]
# From the result, all the rows with null flow bytes have 0 duration, whether it is Normal or DDoS attack


In [ ]:
# Because this is not accidental, i will deal with thiss issue by finding a flag to represent the missing values
# So that I can scale without errors
# However, I may have to create a boolean column with 1/0 for missing/non missing flow bytes values
IDS_dataset['flow_bytes_was_missing'] = IDS_dataset['Flow Bytess'].isnull().astype(int)

# Check the work
print(IDS_dataset['flow_bytes_was_missing'].value_counts())



In [ ]:
# # The value of Flow Bytes for those 77 rows is null as shown below (It is the flow duration that is 0)
# IDS_dataset[IDS_dataset['flow_bytes_was_missing'] == 1]['Flow Bytess'].head()

# To perform calculations on this column, I changed the values to 0
# For reference later, I can always check the 'flow_bytes_was_missing' column to know what was originally missing and what wasn't
IDS_dataset['Flow Bytess'] = IDS_dataset['Flow Bytess'].fillna(0)

# Check the work
print(IDS_dataset['Flow Bytess'].isnull().sum())
# I have dealt with all the null values

#### 3. Data Cleaning - Infinite Values

In [ ]:
# There are some infinite values in the data which I discovered on my first attempt to scale
# These caused error while working with the numeric fields

numeric_cols = IDS_dataset.select_dtypes(include=['float64', 'int64']).columns
inf_counts = np.isinf(IDS_dataset[numeric_cols]).sum()
print(inf_counts[inf_counts > 0])

# From the result, there were about 50 infinite values under Flow Bytess and 127 under Flow Packetss
# Anyway, this cell be comess redundant after running the next but I'm keeping it for record purposes

In [ ]:

# Flag rows with inf values before removing them (same logic as the missing-value flag)
IDS_dataset['flow_had_inf'] = np.isinf(IDS_dataset[numeric_cols]).any(axis=1).astype(int)
print(IDS_dataset['flow_had_inf'].value_counts())

# Check if it's concentrated by class, same check as before
print(IDS_dataset[IDS_dataset['flow_had_inf'] == 1]['newLabel'].value_counts())


# Replace inf with 0 to avoid errors
IDS_dataset[numeric_cols] = IDS_dataset[numeric_cols].replace([np.inf, -np.inf], 0)

# Check the work
print(np.isinf(IDS_dataset[numeric_cols]).sum().sum())  # should print 0

#### 3. Data Cleaning - Duplicated Values

In [ ]:
print(IDS_dataset.duplicated().sum())
IDS_dataset = IDS_dataset.drop_duplicates()
# Checked the number of duplicated rows and dropped them

## Split Data

In [ ]:
# Split the data into y(independent) and x (dependent) variables
x = IDS_dataset.drop(columns=['newLabel'])
y = IDS_dataset['newLabel']

# SPlit the data into train and test while maintaining balance in the proportions of the classes in train/test
# I did this because of the disproportionate representation of classes we noted above
x_train, x_test, y_train, y_test = train_test_split(
    x, y, train_size=0.8, random_state=7, stratify=y
)

## Scale Data

In [ ]:

scaler = StandardScaler()

# I saved the numeric columns to this variable
cols = x_train.select_dtypes(include=['float64', 'int64']).columns

# Non-numeric columns
non_numeric_cols = x_train.select_dtypes(exclude=['float64', 'int64']).columns

# Transform both train and test data
sc_train = scaler.fit_transform(x_train[cols])
sc_test = scaler.transform(x_test[cols])
# I did not fit (learn from) test data in order to avoid data leakage

sc_traindf = pd.DataFrame(sc_train, columns=cols, index=x_train.index)
sc_testdf = pd.DataFrame(sc_test, columns=cols, index=x_test.index)

# Add the non-numeric columns back
sc_traindf = pd.concat([sc_traindf, x_train[non_numeric_cols]], axis=1)
sc_testdf = pd.concat([sc_testdf, x_test[non_numeric_cols]], axis=1)


## Encode Data

In [ ]:
le = LabelEncoder()

y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

print(dict(zip(le.classes_, le.transform(le.classes_))))

In [ ]:
# Reassigning data for ease of remembering

X = sc_traindf.copy()
Y = y_train_encoded.copy()

X_TEST = sc_testdf.copy()
Y_TEST = y_test_encoded.copy()

## Feature Selection

In [ ]:

rfc = RandomForestClassifier(random_state=7)
rfc.fit(X, Y)

# I now rank features by their importance
# That is how much the contribute to the classifier
importances = pd.DataFrame({
    'feature': X.columns,
    'importance': np.round(rfc.feature_importances_, 3)
}).sort_values('importance', ascending=False).set_index('feature')

# Plot only top 20 due to space limitation
importances.head(20).plot.bar(figsize=(12, 4))
plt.show()

In [ ]:
# Select the top 20 features

top_features = importances.head(20).index.tolist()

# Preview selected features
print(top_features)

X = X[top_features]
X_TEST = X_TEST[top_features]

## Model Training

In [ ]:
# KNN
KNN_Classifier = KNeighborsClassifier(n_jobs=-1)
KNN_Classifier.fit(X, Y)

# Logistic Regression
LGR_Classifier = LogisticRegression(n_jobs=-1, random_state=7)
LGR_Classifier.fit(X, Y)

# Bernoulli Naive Bayes
BNB_Classifier = BernoulliNB()
BNB_Classifier.fit(X, Y)

# Decision Tree
DTC_Classifier = tree.DecisionTreeClassifier( criterion='entropy', class_weight='balanced', max_depth=15, min_samples_leaf=3, random_state=7 )
# DTC_Classifier = tree.DecisionTreeClassifier(criterion='entropy', random_state=7)
DTC_Classifier.fit(X, Y)

## Model Evaluation

In [ ]:
models = []
models.append(('Naive Bayes', BNB_Classifier))
models.append(('Decision Tree', DTC_Classifier))
models.append(('KNN', KNN_Classifier))
models.append(('Logistic Regression', LGR_Classifier))

# Evaluate on training data (with cross-validation)
for name, model in models:
    cv_scores = cross_val_score(model, X, Y, cv=10)
    accuracy = metrics.accuracy_score(Y, model.predict(X))
    conf_matrix = metrics.confusion_matrix(Y, model.predict(X))
    report = metrics.classification_report(Y, model.predict(X))

    print(f"\n===== {name} - Train Evaluation =====")
    print("Cross Validation Mean Score:", cv_scores.mean())
    print("Accuracy:", accuracy)
    print("Confusion Matrix:\n", conf_matrix)
    print("Classification Report:\n", report)

## Model Validation

In [ ]:
# Validate the model on test data

for name, model in models:
    accuracy = metrics.accuracy_score(Y_TEST, model.predict(X_TEST))
    conf_matrix = metrics.confusion_matrix(Y_TEST, model.predict(X_TEST))
    report = metrics.classification_report(Y_TEST, model.predict(X_TEST))

    print(f"\n===== {name} - Validation Results =====")
    print("Accuracy:", accuracy)
    print("Confusion Matrix:\n", conf_matrix)
    print("Classification Report:\n", report)


# # To do row by row inspection later, I am, saving this here

# pred_knn = KNN_Classifier.predict(test_df)
# pred_NB = BNB_Classifier.predict(test_df)
# pred_log = LGR_Classifier.predict(test_df)
# pred_dt = DTC_Classifier.predict(test_df)


# DL Model Training

In [23]:
num_classes = len(le.classes_)
Y_train_onehot = to_categorical(Y, num_classes=num_classes)
Y_test_onehot = to_categorical(Y_TEST, num_classes=num_classes)

# Initialising the ANN
classifier = Sequential()

# Adding the input layer and the first hidden layer
classifier.add(Dense(units=8, kernel_initializer='uniform', activation='relu', input_dim=X.shape[1]))

# Adding the second hidden layer
classifier.add(Dense(units=8, kernel_initializer='uniform', activation='relu'))

# Adding the output layer
classifier.add(Dense(units=num_classes, kernel_initializer='uniform', activation='softmax'))

# Compiling the ANN
classifier.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# To fit the ANN to the training data
classifier.fit(X, Y_train_onehot, batch_size=10, epochs=10)

Epoch 1/10
7340/7340 ━━━━━━━━━━━━━━━━━━━━ 16s 2ms/step - accuracy: 0.8211 - loss: 0.5105
Epoch 2/10
7340/7340 ━━━━━━━━━━━━━━━━━━━━ 15s 2ms/step - accuracy: 0.9063 - loss: 0.2893
Epoch 3/10
7340/7340 ━━━━━━━━━━━━━━━━━━━━ 15s 2ms/step - accuracy: 0.9230 - loss: 0.2491
Epoch 4/10
7340/7340 ━━━━━━━━━━━━━━━━━━━━ 14s 2ms/step - accuracy: 0.9302 - loss: 0.2258
Epoch 5/10
7340/7340 ━━━━━━━━━━━━━━━━━━━━ 14s 2ms/step - accuracy: 0.9319 - loss: 0.2123
Epoch 6/10
7340/7340 ━━━━━━━━━━━━━━━━━━━━ 14s 2ms/step - accuracy: 0.9329 - loss: 0.2031
Epoch 7/10
7340/7340 ━━━━━━━━━━━━━━━━━━━━ 14s 2ms/step - accuracy: 0.9345 - loss: 0.1965
Epoch 8/10
7340/7340 ━━━━━━━━━━━━━━━━━━━━ 15s 2ms/step - accuracy: 0.9359 - loss: 0.1917
Epoch 9/10
7340/7340 ━━━━━━━━━━━━━━━━━━━━ 21s 2ms/step - accuracy: 0.9367 - loss: 0.1879
Epoch 10/10
7340/7340 ━━━━━━━━━━━━━━━━━━━━ 15s 2ms/step - accuracy: 0.9379 - loss: 0.1845


In [24]:
# Prediction

yhat_train = classifier.predict(X).argmax(axis=1)
yhat_test = classifier.predict(X_TEST).argmax(axis=1)

2294/2294 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step
574/574 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step


In [25]:
# ANN Evaluation
accuracy = metrics.accuracy_score(Y, yhat_train)
confusion_matrix = metrics.confusion_matrix(Y, yhat_train)
classification = metrics.classification_report(Y, yhat_train)
print()
print('============================== ANN Model Evaluation ==============================')
print()
print("Model Accuracy:" "\n", accuracy)
print()
print("Confusion matrix:" "\n", confusion_matrix)
print()
print("Classification report:" "\n", classification)
print()


============================== ANN Model Evaluation ==============================

Model Accuracy:
 0.9364534368826214

Confusion matrix:
 [[ 1137     0     0     0   358     0     3]
 [    0  7211   913     0    13     7    17]
 [    4    51 20426     0   108    40   165]
 [    0     0     2     0    16     0     5]
 [  155    95  1814     0 18330   320   224]
 [    1     0    16     0    86 20135    89]
 [    0    78    20     0    64     0  1492]]

Classification report:
               precision    recall  f1-score   support

           0       0.88      0.76      0.81      1498
           1       0.97      0.88      0.92      8161
           2       0.88      0.98      0.93     20794
           3       0.00      0.00      0.00        23
           4       0.97      0.88      0.92     20938
           5       0.98      0.99      0.99     20327
           6       0.75      0.90      0.82      1654

    accuracy                           0.94     73395
   macro avg       0.77      0

In [26]:
# ANN Evaluation on TEST set (held-out data)
accuracy_test = metrics.accuracy_score(Y_TEST, yhat_test)
confusion_matrix_test = metrics.confusion_matrix(Y_TEST, yhat_test)
classification_test = metrics.classification_report(Y_TEST, yhat_test)
print()
print('============================== ANN Model Evaluation (TEST) ==============================')
print()
print("Model Accuracy:" "\n", accuracy_test)
print()
print("Confusion matrix:" "\n", confusion_matrix_test)
print()
print("Classification report:" "\n", classification_test)
print()


============================== ANN Model Evaluation (TEST) ==============================

Model Accuracy:
 0.9389612512943485

Confusion matrix:
 [[ 283    0    0    0   90    0    2]
 [   0 1818  217    0    1    3    1]
 [   1   11 5120    0   25    7   34]
 [   0    0    1    0    5    0    0]
 [  37   22  429    0 4612   81   54]
 [   0    0    3    0   25 5035   19]
 [   1   25    6    0   20    0  361]]

Classification report:
               precision    recall  f1-score   support

           0       0.88      0.75      0.81       375
           1       0.97      0.89      0.93      2040
           2       0.89      0.98      0.93      5198
           3       0.00      0.00      0.00         6
           4       0.97      0.88      0.92      5235
           5       0.98      0.99      0.99      5082
           6       0.77      0.87      0.82       413

    accuracy                           0.94     18349
   macro avg       0.78      0.77      0.77     18349
weighted avg      

## CNN

In [ ]:
from keras.layers import Conv1D, MaxPooling1D, Flatten, Dropout, BatchNormalization

# Reshape input for Conv1D: (samples, features, channels)
X_cnn = np.expand_dims(X.values, axis=2)
X_TEST_cnn = np.expand_dims(X_TEST.values, axis=2)

# Build the CNN
cnn_classifier = Sequential()

cnn_classifier.add(Conv1D(32, kernel_size=3, activation='relu', padding='same',
                           input_shape=(X_cnn.shape[1], 1)))
cnn_classifier.add(BatchNormalization())
cnn_classifier.add(MaxPooling1D(pool_size=2))
cnn_classifier.add(Dropout(0.2))

cnn_classifier.add(Conv1D(64, kernel_size=3, activation='relu', padding='same'))
cnn_classifier.add(BatchNormalization())
cnn_classifier.add(MaxPooling1D(pool_size=2))
cnn_classifier.add(Dropout(0.2))

cnn_classifier.add(Flatten())
cnn_classifier.add(Dense(64, activation='relu'))
cnn_classifier.add(Dropout(0.3))
cnn_classifier.add(Dense(num_classes, activation='softmax'))  # multiclass, matches your ANN

cnn_classifier.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
cnn_classifier.summary()

# Train, validating on the actual held-out test set (not a blind validation_split)
cnn_history = cnn_classifier.fit(
    X_cnn, Y_train_onehot,
    validation_data=(X_TEST_cnn, Y_test_onehot),
    batch_size=32, epochs=20, verbose=1
)

# Plot training/validation curves. I did this in order to visualise over fitting

# fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# axes[0].plot(cnn_history.history['accuracy'], label='Train')
# axes[0].plot(cnn_history.history['val_accuracy'], label='Validation')
# axes[0].set_title('CNN Accuracy')
# axes[0].set_xlabel('Epoch')
# axes[0].legend()

# axes[1].plot(cnn_history.history['loss'], label='Train')
# axes[1].plot(cnn_history.history['val_loss'], label='Validation')
# axes[1].set_title('CNN Loss')
# axes[1].set_xlabel('Epoch')
# axes[1].legend()

# plt.tight_layout()
# plt.show()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 20, 32)         │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 20, 32)         │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 10, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 10, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 10, 64)         │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 10, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 5, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 5, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 320)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │        20,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 7)              │           455 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 27,719 (108.28 KB)

 Trainable params: 27,527 (107.53 KB)

 Non-trainable params: 192 (768.00 B)

Epoch 1/20
2294/2294 ━━━━━━━━━━━━━━━━━━━━ 19s 7ms/step - accuracy: 0.8876 - loss: 0.3332 - val_accuracy: 0.9369 - val_loss: 0.1974
Epoch 2/20
2294/2294 ━━━━━━━━━━━━━━━━━━━━ 15s 7ms/step - accuracy: 0.9300 - loss: 0.2172 - val_accuracy: 0.9421 - val_loss: 0.1606
Epoch 3/20
2294/2294 ━━━━━━━━━━━━━━━━━━━━ 15s 6ms/step - accuracy: 0.9385 - loss: 0.1923 - val_accuracy: 0.9591 - val_loss: 0.1397
Epoch 4/20
2294/2294 ━━━━━━━━━━━━━━━━━━━━ 15s 7ms/step - accuracy: 0.9440 - loss: 0.1749 - val_accuracy: 0.9619 - val_loss: 0.1310
Epoch 5/20
2294/2294 ━━━━━━━━━━━━━━━━━━━━ 21s 7ms/step - accuracy: 0.9490 - loss: 0.1627 - val_accuracy: 0.9646 - val_loss: 0.1189
Epoch 6/20
2294/2294 ━━━━━━━━━━━━━━━━━━━━ 16s 7ms/step - accuracy: 0.9515 - loss: 0.1567 - val_accuracy: 0.9523 - val_loss: 0.1370
Epoch 7/20
2294/2294 ━━━━━━━━━━━━━━━━━━━━ 16s 7ms/step - accuracy: 0.9547 - loss: 0.1487 - val_accuracy: 0.9627 - val_loss: 0.1241
Epoch 8/20
2294/2294 ━━━━━━━━━━━━━━━━━━━━ 15s 7ms/step - accuracy: 0.9575 - loss: 0

In [ ]:
# Predict
yhat_train_cnn = cnn_classifier.predict(X_cnn).argmax(axis=1)
yhat_test_cnn = cnn_classifier.predict(X_TEST_cnn).argmax(axis=1)

# CNN Evaluation on TRAIN
accuracy = metrics.accuracy_score(Y, yhat_train_cnn)
confusion_matrix_cnn = metrics.confusion_matrix(Y, yhat_train_cnn)
classification_cnn = metrics.classification_report(Y, yhat_train_cnn)
print()
print('============================== CNN Model Evaluation ==============================')
print()
print("Model Accuracy:" "\n", accuracy)
print()
print("Confusion matrix:" "\n", confusion_matrix_cnn)
print()
print("Classification report:" "\n", classification_cnn)
print()

# CNN Evaluation on TEST set (held-out data)
accuracy_test = metrics.accuracy_score(Y_TEST, yhat_test_cnn)
confusion_matrix_test = metrics.confusion_matrix(Y_TEST, yhat_test_cnn)
classification_test = metrics.classification_report(Y_TEST, yhat_test_cnn)
print()
print('============================== CNN Model Evaluation (TEST) ==============================')
print()
print("Model Accuracy:" "\n", accuracy_test)
print()
print("Confusion matrix:" "\n", confusion_matrix_test)
print()
print("Classification report:" "\n", classification_test)
print()